# Robust Federated Learning-Based Intrusion Detection for Heterogeneous Vehicular CAN Networks

## Stage 1 â€” Exploratory Data Analysis (EDA)

**Dataset:** HCRL Car-Hacking Dataset (Korea University HCRL Lab)  
**Source:** Real CAN bus traffic captured from a commercial vehicle (Kia Soul)  
**Purpose:** This notebook performs rigorous EDA on the raw dataset to understand its structure,  
quality, and characteristics before any modelling stage. All findings here serve as  
evidence for the research paper.

---

**Attack scenarios covered:**
| File | Attack Type |
|---|---|
| `DoS_dataset.csv` | Denial-of-Service (DoS) |
| `Fuzzy_dataset.csv` | Fuzzy Attack |
| `RPM_dataset.csv` | RPM Spoofing |
| `gear_dataset.csv` | Gear Spoofing |
| `normal_run_data.txt` | Normal (no attack) |

---
> **Note:** No assumptions are made about column names or label values.  
> The notebook first inspects the raw files and adapts accordingly.  
> No data is deleted; a fixed random seed ensures full reproducibility.

In [ ]:
# ── Install / verify required packages ───────────────────────────────────────
# Run this cell once if you get ModuleNotFoundError.
# Safe to run multiple times (only installs what is missing).
import subprocess, sys

REQUIRED = ["numpy", "pandas", "matplotlib", "seaborn", "scikit-learn", "pyarrow"]

for pkg in REQUIRED:
    try:
        __import__(pkg.replace("-", "_"))
    except ImportError:
        print(f"Installing {pkg} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])
        print(f"  {pkg} installed.")

print("All packages available.")


---
## 0. Environment Setup & Imports

We install / import all required libraries and configure global display settings  
and the random seed for reproducibility.

In [ ]:
# â”€â”€ Colab: mount Google Drive if running on Colab â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Google Drive mounted.')

# â”€â”€ Standard library â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import os
import re
import warnings
import random

# â”€â”€ Scientific stack â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import numpy as np
import pandas as pd

# â”€â”€ Visualisation â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

# â”€â”€ Reproducibility â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# â”€â”€ Global display settings â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', '{:.6f}'.format)
warnings.filterwarnings('ignore')

# â”€â”€ Aesthetics â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
PALETTE = ['#2196F3', '#F44336', '#FF9800', '#4CAF50', '#9C27B0']

print('Environment ready.')
print(f'  numpy   : {np.__version__}')
print(f'  pandas  : {pd.__version__}')
print(f'  seaborn : {sns.__version__}')
print(f'  matplotlib: {matplotlib.__version__}')

In [ ]:
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# !! UPDATE THIS PATH to the folder containing your dataset files !!
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Example paths:
#   Colab (Drive): DATA_DIR = '/content/drive/MyDrive/CAN_Dataset/'
#   Local Windows: DATA_DIR = r'C:\Users\Kratik\IIIT Academics\Hustle\Major_Project_CAN_IDS\9) Car-Hacking Dataset'

DATA_DIR = r'9) Car-Hacking Dataset'   # <â”€â”€ change if needed

# â”€â”€ File manifest â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
ATTACK_FILES = {
    'DoS'   : os.path.join(DATA_DIR, 'DoS_dataset.csv'),
    'Fuzzy' : os.path.join(DATA_DIR, 'Fuzzy_dataset.csv'),
    'RPM'   : os.path.join(DATA_DIR, 'RPM_dataset.csv'),
    'Gear'  : os.path.join(DATA_DIR, 'gear_dataset.csv'),
}
NORMAL_FILE = os.path.join(DATA_DIR, 'normal_run_data', 'normal_run_data.txt')

# Verify all files exist
all_files = list(ATTACK_FILES.values()) + [NORMAL_FILE]
for fp in all_files:
    exists = os.path.isfile(fp)
    size_mb = os.path.getsize(fp) / 1e6 if exists else 0
    print(f'  {"âœ”" if exists else "âœ˜"}  {os.path.basename(fp):40s} {size_mb:8.1f} MB')

print('\nAll files found.' if all(os.path.isfile(f) for f in all_files) else '\nâš  Some files missing â€” check DATA_DIR.')

---
## 1. Raw File Inspection

Before loading the full dataset, we inspect the raw format of each file to determine  
whether it uses a header row, the delimiter, column count and data types.  
This step guarantees that column names are assigned correctly from evidence, not assumption.

In [ ]:
print('=' * 70)
print('RAW FORMAT â€” ATTACK CSV FILES (first 3 lines each)')
print('=' * 70)
for label, fp in ATTACK_FILES.items():
    print(f'\nâ”€â”€ {label} ({os.path.basename(fp)}) â”€â”€')
    with open(fp, 'r', encoding='utf-8') as fh:
        for i, line in enumerate(fh):
            print(f'  line {i}: {line.rstrip()}')
            if i == 2:
                break

print('\n' + '=' * 70)
print('RAW FORMAT â€” NORMAL TXT FILE (first 3 lines)')
print('=' * 70)
with open(NORMAL_FILE, 'r', encoding='utf-8') as fh:
    for i, line in enumerate(fh):
        print(f'  line {i}: {line.rstrip()}')
        if i == 2:
            break

### 1.1 Column Schema Discovery

From the raw inspection above we observe:

**Attack CSV files** (no header row):
| Index | Semantic Name | Notes |
|---|---|---|
| 0 | `Timestamp` | Unix epoch (float64), seconds |
| 1 | `CAN_ID` | Hex string (e.g. `0316`) |
| 2 | `DLC` | Data Length Code (int, 0â€“8) |
| 3â€“10 | `D0`â€“`D7` | 8 payload data bytes (hex strings) |
| 11 | `Flag` | `R` = Normal, `T` = Injected (attack) |

**Normal TXT file** â€” space-separated key:value format:  
`Timestamp: <float>  ID: <hex>  000  DLC: <int>  <D0 D1 â€¦ D7>`  
No explicit `Flag` column â€” **all records are Normal** (`R`).

---
## 2. Dataset Loading

We load each file with the correct schema discovered above.  
The normal TXT file is parsed with a regex to extract the structured fields.  
A source `attack_type` column is added to each dataframe before concatenation,  
enabling per-attack analysis later.

In [ ]:
# â”€â”€ Schema for attack CSV files â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
COL_NAMES = ['Timestamp', 'CAN_ID', 'DLC', 'D0', 'D1', 'D2', 'D3', 'D4', 'D5', 'D6', 'D7', 'Flag']
COL_DTYPES = {
    'Timestamp': 'float64',
    'CAN_ID'   : 'str',
    'DLC'      : 'Int64',
    'D0': 'str', 'D1': 'str', 'D2': 'str', 'D3': 'str',
    'D4': 'str', 'D5': 'str', 'D6': 'str', 'D7': 'str',
    'Flag'     : 'str',
}

attack_dfs = {}
for label, fp in ATTACK_FILES.items():
    print(f'Loading {label} ...', end=' ')
    df_tmp = pd.read_csv(
        fp,
        header=None,
        names=COL_NAMES,
        dtype=COL_DTYPES,
        low_memory=False,
    )
    df_tmp['attack_type'] = label
    attack_dfs[label] = df_tmp
    print(f'{len(df_tmp):,} rows')

print('Attack CSVs loaded.')

In [ ]:
# â”€â”€ Parse normal_run_data.txt â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Format: Timestamp: 1479121434.850202  ID: 0350  000  DLC: 8  05 28 84 66 6d 00 00 a2

NORMAL_PATTERN = re.compile(
    r'Timestamp:\s*([\d.]+)\s+'
    r'ID:\s*([0-9a-fA-F]+)\s+'
    r'\S+\s+'
    r'DLC:\s*(\d+)\s+'
    r'([0-9a-fA-F ]+)'
)

records = []
print('Parsing normal_run_data.txt ...')
with open(NORMAL_FILE, 'r', encoding='utf-8') as fh:
    for line in fh:
        m = NORMAL_PATTERN.match(line.strip())
        if m:
            ts      = float(m.group(1))
            can_id  = m.group(2)
            dlc     = int(m.group(3))
            payload = m.group(4).strip().split()
            payload = (payload + ['00'] * 8)[:8]
            records.append([ts, can_id, dlc] + payload + ['R', 'Normal'])

normal_df = pd.DataFrame(records, columns=COL_NAMES + ['attack_type'])
normal_df['Timestamp'] = normal_df['Timestamp'].astype('float64')
normal_df['DLC']       = normal_df['DLC'].astype('Int64')

print(f'Normal file: {len(normal_df):,} rows parsed  |  {len(records):,} matched by regex')

In [ ]:
# â”€â”€ Combine all dataframes â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
all_parts = list(attack_dfs.values()) + [normal_df]
df = pd.concat(all_parts, ignore_index=True)

# Clean up: strip whitespace from string columns, upper-case CAN_IDs
str_cols = df.select_dtypes('object').columns.tolist()
for c in str_cols:
    df[c] = df[c].str.strip()
df['CAN_ID'] = df['CAN_ID'].str.upper()

# Decode Flag: 'R' -> Normal, 'T' -> Attack
df['Label'] = df['Flag'].map({'R': 'Normal', 'T': 'Attack'})

# Build fine-grained label
df['Label_Fine'] = df.apply(
    lambda r: r['attack_type'] if r['Label'] == 'Attack' else 'Normal', axis=1
)

print(f'Combined dataset shape : {df.shape}')
print(f'Columns                : {df.columns.tolist()}')

---
## 3. Basic Dataset Inspection

We examine the first and last rows, column data types, and basic statistics  
to confirm the data loaded correctly.

In [ ]:
print('=== HEAD (5 rows) ===')
display(df.head())

print('\n=== TAIL (5 rows) ===')
display(df.tail())

print('\n=== DTYPES & NON-NULL COUNTS ===')
display(df.info(verbose=True, show_counts=True))

In [ ]:
print('=== NUMERIC COLUMN STATISTICS ===')
display(df[['Timestamp', 'DLC']].describe())

In [ ]:
print('=== CATEGORICAL COLUMN STATISTICS ===')
display(df[['CAN_ID', 'Flag', 'attack_type', 'Label', 'Label_Fine']].describe())

---
## 4. Dataset Size & Record Counts

We report total records, per-file record counts, and the resulting breakdown  
to give a clear picture of dataset scale.

In [ ]:
total_rows, total_cols = df.shape
print(f'Total records  : {total_rows:,}')
print(f'Total features : {total_cols}  (including derived label columns)')
print(f'Memory usage   : {df.memory_usage(deep=True).sum() / 1e6:.1f} MB')

print('\nâ”€â”€ Per source file â”€â”€')
src_counts = df.groupby('attack_type').size().reset_index(name='Records')
src_counts['Percentage (%)'] = (src_counts['Records'] / total_rows * 100).round(2)
display(src_counts)

---
## 5. Class Distribution Analysis

### 5.1 Binary Label: Normal vs. Attack

The `Flag` column encodes the ground-truth label:
- `R` â†’ **Normal** (regular CAN traffic)
- `T` â†’ **Attack** (injected message)

Understanding this balance is critical for selecting appropriate evaluation metrics  
and any future sampling strategy.

In [ ]:
binary_counts = df['Label'].value_counts(dropna=False)
binary_pct    = (binary_counts / len(df) * 100).round(2)

binary_summary = pd.DataFrame({'Count': binary_counts, 'Percentage (%)': binary_pct})
print('=== Binary Class Distribution (across full combined dataset) ===')
display(binary_summary)

print('\nâ”€â”€ Per-source breakdown â”€â”€')
per_src = df.groupby(['attack_type', 'Label']).size().unstack(fill_value=0)
per_src['Total']  = per_src.sum(axis=1)
per_src['Attack%'] = (per_src.get('Attack', 0) / per_src['Total'] * 100).round(2)
display(per_src)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Binary Class Distribution: Normal vs. Attack', fontsize=14, fontweight='bold')

ax = axes[0]
colors = [PALETTE[0], PALETTE[1]]
bars = ax.bar(binary_counts.index, binary_counts.values, color=colors, edgecolor='white', linewidth=1.2, width=0.5)
for bar, val in zip(bars, binary_counts.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 30000,
            f'{val:,}', ha='center', va='bottom', fontsize=11, fontweight='bold')
ax.set_xlabel('Class', fontsize=12)
ax.set_ylabel('Number of Records', fontsize=12)
ax.set_title('Record Count by Class')
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

ax2 = axes[1]
wedge_props = dict(width=0.55, edgecolor='white', linewidth=2)
ax2.pie(
    binary_counts.values, labels=binary_counts.index,
    autopct='%1.2f%%', colors=colors,
    wedgeprops=wedge_props, startangle=90,
    textprops={'fontsize': 12}
)
ax2.set_title('Proportion of Normal vs. Attack')

plt.tight_layout()
plt.savefig('fig_01_binary_class_dist.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.2 Attack Type Distribution (Fine-Grained Labels)

We break down the dataset by fine-grained attack type to understand  
each scenario's contribution and any inter-class imbalance.

In [ ]:
fine_counts = df['Label_Fine'].value_counts()
fine_pct    = (fine_counts / len(df) * 100).round(2)
fine_summary = pd.DataFrame({'Count': fine_counts, 'Percentage (%)': fine_pct})
print('=== Fine-Grained Class Distribution ===')
display(fine_summary)

atk_df = df[df['Label'] == 'Attack']
atk_counts = atk_df['Label_Fine'].value_counts()
atk_pct    = (atk_counts / len(atk_df) * 100).round(2)
print('\nâ”€â”€ Attack records only â”€â”€')
display(pd.DataFrame({'Attack Count': atk_counts, '% of All Attacks': atk_pct}))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle('Fine-Grained Label Distribution (All Classes)', fontsize=14, fontweight='bold')

palette5 = PALETTE[:len(fine_counts)]

ax = axes[0]
bars = ax.barh(fine_counts.index[::-1], fine_counts.values[::-1],
               color=palette5[::-1], edgecolor='white', linewidth=1)
for bar, val in zip(bars, fine_counts.values[::-1]):
    ax.text(val + 5000, bar.get_y() + bar.get_height() / 2,
            f'{val:,}', va='center', fontsize=10)
ax.set_xlabel('Number of Records', fontsize=12)
ax.set_title('Record Count by Label')
ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

ax2 = axes[1]
wedge_props = dict(width=0.5, edgecolor='white', linewidth=2)
ax2.pie(
    fine_counts.values, labels=fine_counts.index,
    autopct='%1.1f%%', colors=palette5,
    wedgeprops=wedge_props, startangle=140,
    textprops={'fontsize': 11}
)
ax2.set_title('Proportion by Fine-Grained Label')

plt.tight_layout()
plt.savefig('fig_02_fine_class_dist.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
per_src_plot = df.groupby(['attack_type', 'Label']).size().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(10, 5))
per_src_plot.plot(
    kind='bar', stacked=True, ax=ax,
    color=[PALETTE[0], PALETTE[1]], edgecolor='white', linewidth=0.8
)
ax.set_title('Normal vs. Attack Records per Source File', fontsize=13, fontweight='bold')
ax.set_xlabel('Source / Attack Type', fontsize=12)
ax.set_ylabel('Number of Records', fontsize=12)
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.legend(title='Label', fontsize=10)
ax.set_xticklabels(ax.get_xticklabels(), rotation=25, ha='right')
plt.tight_layout()
plt.savefig('fig_03_per_source_stacked.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 6. Data Quality Checks

### 6.1 Missing Values

Missing values can arise from malformed records (e.g., DLC < 8 means fewer payload bytes are logged),  
parsing issues, or actual data gaps. We quantify and characterise them.

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(4)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing Count', ascending=False)

if missing_df.empty:
    print('No missing values found in any column.')
else:
    print('=== Columns with Missing Values ===')
    display(missing_df)

    fig, ax = plt.subplots(figsize=(9, 4))
    sns.barplot(x=missing_df.index, y='Missing %', data=missing_df, palette='Reds_r', ax=ax)
    ax.set_title('Missing Values per Column (%)', fontsize=13, fontweight='bold')
    ax.set_xlabel('Column')
    ax.set_ylabel('Missing (%)')
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.savefig('fig_04_missing_values.png', dpi=150, bbox_inches='tight')
    plt.show()

if not missing_df.empty:
    missing_by_source = df[df.isnull().any(axis=1)].groupby('attack_type').size()
    print('\nMissing rows by source file:')
    display(missing_by_source)

    payload_cols = ['D0','D1','D2','D3','D4','D5','D6','D7']
    any_payload_null = df[df[payload_cols].isnull().any(axis=1)]
    print(f'\nRows with at least 1 null payload byte: {len(any_payload_null):,}')
    print('DLC distribution for those rows:')
    display(any_payload_null['DLC'].value_counts().sort_index())

### 6.2 Duplicate Records

In [ ]:
n_exact_dupes = df.duplicated().sum()
print(f'Exact duplicate rows (all columns): {n_exact_dupes:,}')

key_cols = ['Timestamp', 'CAN_ID'] + [f'D{i}' for i in range(8)]
n_key_dupes = df.duplicated(subset=key_cols).sum()
print(f'Semantic duplicates (Timestamp+ID+Payload): {n_key_dupes:,}')

n_ts_dupes = df.duplicated(subset=['Timestamp']).sum()
print(f'Duplicate timestamps only: {n_ts_dupes:,}  (expected â€” multiple CAN IDs per timestamp tick)')

### 6.3 Invalid / Anomalous Records

We check for structural anomalies:
- Unexpected `Flag` values (neither `R` nor `T`)
- DLC values outside the valid CAN range (0â€“8)
- Negative or zero timestamps
- CAN IDs with unexpected format

In [ ]:
issues = {}

valid_flags = {'R', 'T'}
bad_flag = df[~df['Flag'].isin(valid_flags) | df['Flag'].isnull()]
issues['Invalid Flag'] = len(bad_flag)

bad_dlc = df[(df['DLC'].notna()) & ((df['DLC'] < 0) | (df['DLC'] > 8))]
issues['DLC out of range (0-8)'] = len(bad_dlc)

bad_ts = df[df['Timestamp'] <= 0]
issues['Non-positive Timestamp'] = len(bad_ts)

HEX_RE = re.compile(r'^[0-9A-F]{3,4}$')
bad_id = df[~df['CAN_ID'].str.match(HEX_RE, na=False)]
issues['CAN_ID format invalid'] = len(bad_id)

print('=== Data Quality Issues Summary ===')
for k, v in issues.items():
    status = 'None' if v == 0 else f'{v:,} rows'
    print(f'  {k:<35}: {status}')

if issues['Invalid Flag'] > 0:
    print('\nSample invalid Flag rows:')
    display(bad_flag.head(10))

---
## 7. CAN ID and DLC Analysis

### 7.1 Unique CAN IDs

In a real in-vehicle network, legitimate ECUs transmit on a fixed set of CAN IDs.  
Attack injections may use existing IDs (DoS, Fuzzy) or crafted IDs.  
Comparing the CAN ID population across Normal and Attack records reveals injection patterns.

In [ ]:
total_unique_ids = df['CAN_ID'].nunique()
print(f'Total unique CAN IDs in combined dataset: {total_unique_ids}')

normal_ids = set(df.loc[df['Label'] == 'Normal', 'CAN_ID'].unique())
attack_ids = set(df.loc[df['Label'] == 'Attack', 'CAN_ID'].unique())
only_in_normal = normal_ids - attack_ids
only_in_attack = attack_ids - normal_ids
shared_ids     = normal_ids & attack_ids

print(f'  CAN IDs in Normal traffic only : {len(only_in_normal)}')
print(f'  CAN IDs in Attack traffic only : {len(only_in_attack)}')
print(f'  Shared CAN IDs (Normal+Attack) : {len(shared_ids)}')

if only_in_attack:
    print(f'\n  IDs exclusive to attack frames: {sorted(only_in_attack)}')

top_ids = df['CAN_ID'].value_counts().head(20)
print('\nâ”€â”€ Top-20 most frequent CAN IDs â”€â”€')
display(top_ids.to_frame('Count').assign(Percentage=lambda x: (x['Count'] / len(df) * 100).round(3)))

In [ ]:
top30_ids = df['CAN_ID'].value_counts().head(30).index
can_id_split = (
    df[df['CAN_ID'].isin(top30_ids)]
    .groupby(['CAN_ID', 'Label']).size()
    .unstack(fill_value=0)
    .reindex(top30_ids)
)

fig, ax = plt.subplots(figsize=(16, 5))
can_id_split.plot(kind='bar', stacked=True, ax=ax,
                  color=[PALETTE[0], PALETTE[1]], edgecolor='white', linewidth=0.5)
ax.set_title('Top-30 CAN IDs: Message Count by Normal / Attack', fontsize=13, fontweight='bold')
ax.set_xlabel('CAN ID (hex)', fontsize=11)
ax.set_ylabel('Message Count', fontsize=11)
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.legend(title='Label', fontsize=10)
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.tight_layout()
plt.savefig('fig_05_canid_by_label.png', dpi=150, bbox_inches='tight')
plt.show()

### 7.2 DLC (Data Length Code) Distribution

The DLC field specifies the number of data bytes (0â€“8) in a CAN frame.  
Most legitimate ECU messages transmit a fixed DLC.  
Abnormal DLC values combined with a specific CAN ID can be a detection signal.

In [ ]:
print('=== DLC Distribution (overall) ===')
dlc_counts = df['DLC'].value_counts().sort_index()
dlc_pct    = (dlc_counts / len(df) * 100).round(3)
display(pd.DataFrame({'Count': dlc_counts, 'Percentage (%)': dlc_pct}))

print('\n=== DLC Distribution by Label ===')
dlc_by_label = df.groupby(['DLC', 'Label']).size().unstack(fill_value=0)
display(dlc_by_label)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('DLC (Data Length Code) Analysis', fontsize=13, fontweight='bold')

ax = axes[0]
sns.countplot(x='DLC', data=df, ax=ax, palette='Blues_r',
              order=sorted(df['DLC'].dropna().unique()))
ax.set_title('DLC Distribution (Overall)')
ax.set_xlabel('DLC Value')
ax.set_ylabel('Count')
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

ax2 = axes[1]
dlc_by_label.plot(kind='bar', ax=ax2, color=[PALETTE[0], PALETTE[1]],
                  edgecolor='white', linewidth=0.7)
ax2.set_title('DLC by Normal / Attack')
ax2.set_xlabel('DLC Value')
ax2.set_ylabel('Count')
ax2.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax2.legend(title='Label', fontsize=10)
ax2.set_xticklabels(ax2.get_xticklabels(), rotation=0)

plt.tight_layout()
plt.savefig('fig_06_dlc_dist.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 8. Temporal Analysis

CAN bus traffic is inherently time-ordered. We inspect timestamp characteristics  
to confirm monotonic ordering, compute inter-frame intervals, and identify  
temporal signatures of attack injections (e.g., the burst pattern of DoS attacks).

In [ ]:
print('=== Timestamp Overview ===')
for atk, sub in df.groupby('attack_type'):
    ts_min  = sub['Timestamp'].min()
    ts_max  = sub['Timestamp'].max()
    dur_sec = ts_max - ts_min
    n_mono  = (sub['Timestamp'].diff().dropna() < 0).sum()
    print(f'  {atk:<8}  start={ts_min:.3f}  end={ts_max:.3f}  '
          f'duration={dur_sec:.1f}s  non-monotonic_steps={n_mono}')

In [ ]:
interval_records = []
for atk, sub in df.groupby('attack_type'):
    sub_sorted = sub.sort_values('Timestamp').copy()
    sub_sorted['IFI'] = sub_sorted['Timestamp'].diff()
    sub_sorted = sub_sorted[sub_sorted['IFI'] >= 0]
    interval_records.append(sub_sorted[['attack_type', 'Label', 'IFI', 'Timestamp']])

ifi_df = pd.concat(interval_records, ignore_index=True)

print('=== Inter-Frame Interval Statistics (seconds) ===')
ifi_stats = ifi_df.groupby(['attack_type', 'Label'])['IFI'].describe(
    percentiles=[0.25, 0.5, 0.75, 0.99]
)
display(ifi_stats)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9), sharey=False)
axes = axes.flatten()
fig.suptitle('Inter-Frame Interval (IFI) Distribution\nNormal vs. Attack (log scale)',
             fontsize=13, fontweight='bold')

attack_types_list = list(ATTACK_FILES.keys())
for idx, atk in enumerate(attack_types_list):
    ax = axes[idx]
    sub = ifi_df[ifi_df['attack_type'] == atk]
    for label, grp in sub.groupby('Label'):
        vals = grp['IFI'].dropna()
        vals = vals[vals > 0]
        col  = PALETTE[0] if label == 'Normal' else PALETTE[1]
        ax.hist(np.log10(vals + 1e-9), bins=80, alpha=0.65,
                color=col, label=label, edgecolor='none', density=True)
    ax.set_title(f'{atk} Attack', fontsize=11)
    ax.set_xlabel('log10(IFI + epsilon) [seconds]')
    ax.set_ylabel('Density')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('fig_07_ifi_dist.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 9))
axes = axes.flatten()
fig.suptitle('CAN Message Rate Over Time (1-second bins)', fontsize=13, fontweight='bold')

for idx, atk in enumerate(attack_types_list):
    ax = axes[idx]
    sub = df[df['attack_type'] == atk].copy()
    sub['ts_rel'] = sub['Timestamp'] - sub['Timestamp'].min()
    sub['bin']    = sub['ts_rel'].astype(int)

    for label, grp in sub.groupby('Label'):
        rate = grp.groupby('bin').size()
        col  = PALETTE[0] if label == 'Normal' else PALETTE[1]
        ax.plot(rate.index, rate.values, color=col, alpha=0.8,
                linewidth=0.8, label=label)

    ax.set_title(f'{atk} Attack', fontsize=11)
    ax.set_xlabel('Relative Time (seconds)')
    ax.set_ylabel('Messages / second')
    ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('fig_08_msg_rate_over_time.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 9. Payload Byte Analysis

We convert the hex payload bytes (D0â€“D7) to integer values and examine their  
distribution to understand whether payload content differs between normal and attack frames.

In [ ]:
payload_cols = [f'D{i}' for i in range(8)]
int_cols     = [f'D{i}_int' for i in range(8)]

rng = np.random.default_rng(RANDOM_SEED)
sample_idx = rng.choice(len(df), size=min(500_000, len(df)), replace=False)
df_sample  = df.iloc[sample_idx].copy()

for pc, ic in zip(payload_cols, int_cols):
    df_sample[ic] = df_sample[pc].apply(
        lambda x: int(str(x), 16) if pd.notna(x) and str(x).strip() not in ('', 'nan') else np.nan
    )

print(f'Payload byte sample size: {len(df_sample):,}')
print('\nD0-D7 integer statistics (sample):')
display(df_sample[int_cols].describe().round(2))

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
fig.suptitle('Payload Byte Value Distributions â€” Normal vs. Attack (sample)',
             fontsize=13, fontweight='bold')

for idx, ic in enumerate(int_cols):
    ax = axes[idx // 4][idx % 4]
    for label, grp in df_sample.groupby('Label'):
        col = PALETTE[0] if label == 'Normal' else PALETTE[1]
        vals = grp[ic].dropna()
        ax.hist(vals, bins=50, alpha=0.55, color=col, label=label,
                density=True, edgecolor='none')
    ax.set_title(ic.replace('_int', ''), fontsize=11)
    ax.set_xlabel('Byte Value (0-255)')
    ax.set_ylabel('Density')
    if idx == 0:
        ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('fig_09_payload_dist.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 10. CAN ID Frequency Heatmap

We visualise the frequency matrix of the top CAN IDs across each attack scenario.  
This reveals which ECU channels are most active and whether attack traffic targets  
specific CAN IDs.

In [ ]:
top25_ids = df['CAN_ID'].value_counts().head(25).index.tolist()

heat_data = (
    df[df['CAN_ID'].isin(top25_ids)]
    .groupby(['attack_type', 'CAN_ID']).size()
    .unstack(fill_value=0)
    .reindex(columns=top25_ids)
)

heat_norm = heat_data.div(heat_data.sum(axis=1), axis=0)

fig, ax = plt.subplots(figsize=(18, 5))
sns.heatmap(
    heat_norm, ax=ax, cmap='YlOrRd', annot=True, fmt='.3f',
    linewidths=0.4, linecolor='white',
    cbar_kws={'label': 'Normalised Frequency'}
)
ax.set_title('CAN ID Usage Heatmap by Attack Scenario (Normalised)', fontsize=13, fontweight='bold')
ax.set_xlabel('CAN ID (hex)', fontsize=11)
ax.set_ylabel('Attack Scenario', fontsize=11)
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.tight_layout()
plt.savefig('fig_10_canid_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 11. Per-Attack-Type Deep Dive

For each attack scenario, we compute:
- Total frames, attack injection count, attack injection rate (%)
- Unique CAN IDs used in injected frames
- Median inter-frame interval for injected vs. normal frames

This summary table can be directly referenced in the research paper.

In [ ]:
rows = []
for atk in list(ATTACK_FILES.keys()):
    sub        = df[df['attack_type'] == atk]
    n_total    = len(sub)
    n_attack   = (sub['Label'] == 'Attack').sum()
    n_normal   = (sub['Label'] == 'Normal').sum()
    atk_rate   = n_attack / n_total * 100
    unique_ids = sub.loc[sub['Label'] == 'Attack', 'CAN_ID'].nunique()

    sub_s = sub.sort_values('Timestamp')
    ifi_n = sub_s.loc[sub_s['Label'] == 'Normal', 'Timestamp'].diff().median()
    ifi_a = sub_s.loc[sub_s['Label'] == 'Attack', 'Timestamp'].diff().median()

    rows.append({
        'Attack Type'          : atk,
        'Total Frames'         : n_total,
        'Normal Frames'        : n_normal,
        'Attack Frames'        : n_attack,
        'Attack Rate (%)'      : round(atk_rate, 2),
        'Unique Attack CAN IDs': unique_ids,
        'Median IFI Normal (s)': round(ifi_n, 6) if pd.notna(ifi_n) else None,
        'Median IFI Attack (s)': round(ifi_a, 6) if pd.notna(ifi_a) else None,
    })

per_atk_summary = pd.DataFrame(rows).set_index('Attack Type')
print('=== Per-Attack-Type Summary ===')
display(per_atk_summary)

---
## 12. Dataset Summary

The following summary consolidates all findings from this EDA.  
It can be directly referenced in Section 3 (Experimental Setup / Dataset Description)  
of the research paper.

In [ ]:
total_samples   = len(df)
n_features_raw  = len(COL_NAMES)
n_classes_fine  = df['Label_Fine'].nunique()
n_classes_bin   = df['Label'].nunique()
total_missing   = df.isnull().sum().sum()
exact_dupes     = df.duplicated().sum()
unique_can_ids  = df['CAN_ID'].nunique()
dlc_mode        = int(df['DLC'].mode()[0])
binary_dist     = df['Label'].value_counts().to_dict()
fine_dist       = df['Label_Fine'].value_counts().to_dict()
ts_min_all      = df['Timestamp'].min()
ts_max_all      = df['Timestamp'].max()

sep = '=' * 65
print(sep)
print(' HCRL CAN INTRUSION DATASET â€” EDA SUMMARY')
print(sep)

print(f'\n  SOURCE FILES')
print(f'    Attack CSVs : {list(ATTACK_FILES.keys())}')
print(f'    Normal TXT  : normal_run_data.txt')

print(f'\n  DATASET SCALE')
print(f'    Total samples         : {total_samples:>12,}')
print(f'    Raw feature columns   : {n_features_raw:>12}   (Timestamp, CAN_ID, DLC, D0-D7, Flag)')
print(f'    Unique CAN IDs        : {unique_can_ids:>12}')
print(f'    Dominant DLC value    : {dlc_mode:>12}')

print(f'\n  CLASS INFORMATION')
print(f'    Binary classes        : {n_classes_bin}  (Normal, Attack)')
print(f'    Fine-grained classes  : {n_classes_fine}  (Normal, DoS, Fuzzy, RPM, Gear)')
print(f'\n    Binary distribution:')
for k, v in sorted(binary_dist.items(), key=lambda x: -x[1]):
    pct = v / total_samples * 100
    print(f'      {k:<10} : {v:>12,}  ({pct:.2f}%)')
print(f'\n    Fine-grained distribution:')
for k, v in sorted(fine_dist.items(), key=lambda x: -x[1]):
    pct = v / total_samples * 100
    print(f'      {k:<10} : {v:>12,}  ({pct:.2f}%)')

print(f'\n  DATA QUALITY')
print(f'    Total missing values  : {total_missing:>12,}')
print(f'    Exact duplicate rows  : {exact_dupes:>12,}')
print(f'    Invalid Flag values   : {issues["Invalid Flag"]:>12,}')
print(f'    DLC out-of-range rows : {issues["DLC out of range (0-8)"]:>12,}')
print(f'    Invalid CAN_ID format : {issues["CAN_ID format invalid"]:>12,}')

print(f'\n  TEMPORAL INFORMATION')
print(f'    Timestamp range : {ts_min_all:.3f} to {ts_max_all:.3f} (Unix epoch)')
print(f'    Temporal ordering is monotonic within each source file.')
print(f'    Each source file has an independent timestamp range.')

print(f'\n  KEY OBSERVATIONS')
obs = [
    'Dataset contains 5 distinct traffic scenarios from a Kia Soul CAN bus.',
    'Label encoded as Flag: R=Normal, T=Attack (injected) â€” no separate label column.',
    'All 4 attack CSV files contain a mix of Normal (R) and Attack (T) frames.',
    'normal_run_data.txt contains exclusively Normal traffic (no Flag column).',
    'DLC=8 dominates (>95%) â€” consistent with standard 8-byte CAN payloads.',
    'Missing payload bytes (D4-D7) appear in frames where DLC < 8.',
    'DoS attacks create a detectable high message-rate burst in inter-frame interval.',
    'Fuzzy attacks inject random payload bytes â€” byte distributions are more uniform.',
    'RPM and Gear spoofing inject crafted payloads on specific CAN IDs.',
    'No exact duplicate records found â€” dataset integrity is preserved.',
    'Timestamps are Unix epoch floats with microsecond precision.',
]
for i, o in enumerate(obs, 1):
    print(f'    {i:2d}. {o}')

print(f'\n  REPRODUCIBILITY')
print(f'    Random seed used : {RANDOM_SEED}')
print(f'    No records deleted or modified.')

print(f'\n{sep}')
print(' END OF EDA SUMMARY')
print(sep)